In [25]:
import os
import random
import sqlite3
from datetime import datetime, timedelta

import pandas as pd
from faker import Faker

In [26]:
# 1. Configuration
DB_PATH = "../data/raw/banque_entreprise.db"
os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)

Faker.seed(42)
random.seed(42)
fake = Faker('fr_FR')
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

In [27]:
# 2. Création du schéma relationnel
cursor.executescript('''
    DROP TABLE IF EXISTS factures;
    DROP TABLE IF EXISTS abonnements;
    DROP TABLE IF EXISTS clients;

    CREATE TABLE clients (
        client_id INTEGER PRIMARY KEY AUTOINCREMENT,
        nom_entreprise TEXT,
        contact_email TEXT,
        segment TEXT,
        region TEXT,
        date_inscription DATE
    );

    CREATE TABLE abonnements (
        abonnement_id INTEGER PRIMARY KEY AUTOINCREMENT,
        client_id INTEGER,
        type_forfait TEXT,
        prix_mensuel REAL,
        statut_actif BOOLEAN,
        FOREIGN KEY (client_id) REFERENCES clients (client_id)
    );

    CREATE TABLE factures (
        facture_id INTEGER PRIMARY KEY AUTOINCREMENT,
        abonnement_id INTEGER,
        montant_paye REAL,
        date_facturation DATE,
        statut_paiement TEXT,
        FOREIGN KEY (abonnement_id) REFERENCES abonnements (abonnement_id)
    );
''')

In [ ]:
# --- MODIF 1 : cas limites déterministes ---
CLIENTS_GMAIL = set(range(100, 5001, 100))              # 50 clients en @gmail.com
CLIENTS_SANS_ABO = set(range(499, 5001, 499))           # 10 clients sans abonnement, inscrits en 2026
CLIENTS_REGION_INACTIVE = set(range(701, 5001, 701))    # 7 clients dans une région sans abonnement actif


# --- MODIF 2 : régions sans abonnements
regions_possibles = sorted({fake.region() for _ in range(2000)})
REGIONS_INACTIVES = ['Limousin', 'Wallis-et-Futuna', 'Nouvelle-Calédonie']
assert all(r in regions_possibles for r in REGIONS_INACTIVES)


REGION_DE_CLIENT = {
    cid: REGIONS_INACTIVES[i % len(REGIONS_INACTIVES)]
    for i, cid in enumerate(sorted(CLIENTS_REGION_INACTIVE))
}

def region_normale():
    r = fake.region()
    while r in REGIONS_INACTIVES: 
        r = fake.region()
    return r


# Variables de distribution métier
segments = ['PME', 'ETI', 'Grand Compte']
forfaits = {
    'Compte Pro Premium': 25.0, 
    'Location Terminal de Paiement (TPE)': 40.0, 
    'Pack Cartes Corporate Gold': 130.0, 
    'Assurance Responsabilité Civile': 210.0
}
statuts_paiement = ['Payée', 'Payée', 'Payée', 'En retard', 'Échouée'] # 60% de chances d'être payée

clients_data = []
abonnements_data = []
factures_data = []

abonnement_counter = 1

# Génération de 5000 clients
for client_id in range(1, 5001):
    # MODIF 3 : clients sans abonnement forcés en 2026
    if client_id in CLIENTS_SANS_ABO:
        date_inscription = fake.date_between(start_date=datetime(2026, 1, 1).date(), end_date='today')
    else:
        date_inscription = fake.date_between(start_date='-3y', end_date='today')
        
    # MODIF 4 : emails gmail
    email = fake.company_email()
    if client_id in CLIENTS_GMAIL:
        email = email.split('@')[0] + '@gmail.com'
    
    # --- MODIF 2.1 : régions réelles sans abonnement actif ---
    region = REGION_DE_CLIENT.get(client_id) or region_normale()    

        
    clients_data.append((
        fake.company(),
        email,
        random.choices(segments, weights=[0.6, 0.3, 0.1])[0],
        region,
        date_inscription.strftime('%Y-%m-%d')
    ))

    # MODIF 5 : 0 abonnement pour certains clients
    nb_abonnements = 0 if client_id in CLIENTS_SANS_ABO else random.randint(1, 3)
    for _ in range(nb_abonnements):
        type_forfait = random.choice(list(forfaits.keys()))
        prix_base = forfaits[type_forfait]
        prix_mensuel = round(prix_base * random.uniform(0.8, 1.0), 2)

        # MODIF 6 : abonnements inactifs dans la région dédiée
        if client_id in CLIENTS_REGION_INACTIVE:
            statut_actif = False
        else:
            statut_actif = random.random() > 0.15

        abonnements_data.append((
            abonnement_counter, client_id, type_forfait, prix_mensuel, statut_actif
        ))

        # MODIF 7 : 1 abonnement sur 200 sans aucune facture
        if abonnement_counter % 200 != 0:
            # MODIF 8 : 12 mois d'historique au lieu de 6
            for mois_recul in range(12):
                date_facture = datetime.today() - timedelta(days=30 * mois_recul)
                if date_facture.date() > date_inscription:
                    factures_data.append((
                        abonnement_counter,
                        prix_mensuel,
                        date_facture.strftime('%Y-%m-%d'),
                        random.choice(statuts_paiement)
                    ))

        abonnement_counter += 1

In [ ]:
cursor.executemany('''
    INSERT INTO clients (nom_entreprise, contact_email, segment, region, date_inscription)
    VALUES (?, ?, ?, ?, ?)
''', clients_data)

cursor.executemany('''
    INSERT INTO abonnements (abonnement_id, client_id, type_forfait, prix_mensuel, statut_actif)
    VALUES (?, ?, ?, ?, ?)
''', abonnements_data)

cursor.executemany('''
    INSERT INTO factures (abonnement_id, montant_paye, date_facturation, statut_paiement)
    VALUES (?, ?, ?, ?)
''', factures_data)

cursor.execute("CREATE INDEX IF NOT EXISTS idx_abonnements_client ON abonnements(client_id)")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_factures_abonnement ON factures(abonnement_id)")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_factures_date ON factures(date_facturation)")


conn.commit()
conn.close()

print(f"Base de données générée avec succès dans {DB_PATH}")
print(f"Total : {len(clients_data)} clients, {len(abonnements_data)} abonnements, {len(factures_data)} factures.")

Base de données générée avec succès dans ../data/raw/banque_entreprise.db
Total : 5000 clients, 10059 abonnements, 102448 factures.


In [30]:
query_clients = "SELECT * FROM clients"
query_factures = "SELECT * FROM factures"
query_abonnements = "SELECT * FROM abonnements"

conn = sqlite3.connect(DB_PATH)
df_clients = pd.read_sql_query(query_clients, conn)
df_factures = pd.read_sql_query(query_factures, conn)
df_abonnements = pd.read_sql_query(query_abonnements, conn)
conn.close()

In [31]:
df_clients.head()

,client_id,nom_entreprise,contact_email,segment,region,date_inscription
0,1,Bernard Boulay SA,marcelleleveque@colin.fr,ETI,Franche-Comté,2024-11-24
1,2,Jacques Perrot S.A.,umaillet@rossi.fr,PME,Nord-Pas-de-Calais,2026-09-16
2,3,Mercier Roussel S.A.S.,davidsuzanne@bonneau.com,ETI,Basse-Normandie,2024-08-07
3,4,Joly S.A.S.,moulinnicole@marechal.net,PME,Midi-Pyrénées,2023-11-15
4,5,Meunier,sylviefoucher@verdier.fr,PME,Basse-Normandie,2026-10-02


In [32]:
df_factures.head()

,facture_id,abonnement_id,montant_paye,date_facturation,statut_paiement
0,1,1,110.37,2026-10-10,Payée
1,2,1,110.37,2026-09-10,Échouée
2,3,1,110.37,2026-08-11,Payée
3,4,1,110.37,2026-07-12,Échouée
4,5,1,110.37,2026-06-12,En retard


In [33]:
df_abonnements.head()

,abonnement_id,client_id,type_forfait,prix_mensuel,statut_actif
0,1,1,Pack Cartes Corporate Gold,110.37,0
1,2,2,Assurance Responsabilité Civile,177.26,1
2,3,3,Assurance Responsabilité Civile,182.29,1
3,4,4,Compte Pro Premium,22.76,1
4,5,4,Pack Cartes Corporate Gold,115.79,1
